# ARC-05 Simulation Package Overview

ARC-05 provides a reusable Python interface for an educational Earth point-mass two-body simulation. It propagates one spacecraft's translational state in an Earth-Centered Inertial (ECI) frame using SI units and fixed-step classical Runge-Kutta 4 (RK4). This is a research and learning baseline, not a mission-grade or certified system.

## Package architecture

- `models.py`: strict `SimulationConfig` and immutable `OrbitalState` data contracts.
- `dynamics.py`: Earth two-body state derivative.
- `integrator.py`: generic RK4 stepper.
- `propagation.py`: trajectory propagation and sampled collision handling.
- `diagnostics.py`: orbital energy, angular momentum, eccentricity, radius, speed, and drift.
- `workflow.py`: `run_simulation`, which joins configuration, propagation, and diagnostics.
- `simulation/__init__.py`: public package exports used by callers.

The state is `[x, y, z, vx, vy, vz]`, with positions in metres, velocities in metres per second, and time in seconds. `SimulationConfig` selects ECI and RK4 and validates finite SI-valued inputs. `OrbitalState` represents one state; `run_simulation` accepts its position and velocity through the configuration and returns sampled times, states, collision information, diagnostics, and model metadata.

## Small public-API example

The next cell locates this repository's `src` layout, creates a short simulation, and reports values returned by the real public workflow. The printed shapes and diagnostics are results of this particular configuration, not accuracy guarantees.

In [1]:
import sys as _notebook_sys
from pathlib import Path as _NotebookPath

_notebook_cwd = _NotebookPath.cwd()
_notebook_repo_root = next(
    (
        candidate
        for candidate in (_notebook_cwd, *_notebook_cwd.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "quantum_space_agi").is_dir()
    ),
    None,
)
if _notebook_repo_root is None:
    raise RuntimeError(
        f"Could not locate repository root from notebook working directory {_notebook_cwd}; "
        "open the notebook from within the repository."
    )
_notebook_src_path = str(_notebook_repo_root / "src")
if _notebook_src_path not in _notebook_sys.path:
    _notebook_sys.path.insert(0, _notebook_src_path)

import quantum_space_agi as _notebook_package
if _NotebookPath(_notebook_package.__file__).resolve().parent != (
    _notebook_repo_root / "src" / "quantum_space_agi"
).resolve():
    raise ImportError("quantum_space_agi did not resolve from this repository's src directory")

from pathlib import Path
import sys

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "quantum_space_agi").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root from the notebook working directory")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
from quantum_space_agi.simulation import OrbitalState, SimulationConfig, run_simulation

In [2]:
example_initial_state = OrbitalState(
    position_m=(7_000_000.0, 0.0, 0.0),
    velocity_mps=(0.0, 7_500.0, 0.0),
    time_s=0.0,
)
example_config = SimulationConfig(
    initial_position_m=example_initial_state.position_m,
    initial_velocity_mps=example_initial_state.velocity_mps,
    simulation_duration_s=600.0,
    time_step_s=10.0,
    collision_radius_m=6_378_137.0,
)
example_result = run_simulation(example_config, initial_time_s=example_initial_state.time_s)

print("Model:", example_result["model"])
print("Time samples:", example_result["times_s"].shape)
print("State samples:", example_result["states"].shape)
print("Final time (s):", example_result["times_s"][-1])
print("Maximum absolute energy drift (m^2/s^2):", example_result["diagnostics"]["max_abs_energy_drift_m2_s2"])
print("Collision time (s):", example_result["collision_time_s"])

Model: {'reference_frame': 'ECI', 'integrator': 'RK4', 'dynamics': 'Earth point-mass Newtonian two-body', 'time_unit': 's', 'position_unit': 'm', 'velocity_unit': 'm/s'}
Time samples: (61,)
State samples: (61, 6)
Final time (s): 600.0
Maximum absolute energy drift (m^2/s^2): 4.485994577407837e-05
Collision time (s): None
